### Read all files from Volume (Bronze ingestion)

In [0]:
df_customers = spark.read.option("header", True).csv(
    "dbfs:/Volumes/dedatabricsws/demo/raw_volume/customers.csv"
)

# df_customers.show()

In [0]:
df_customers.write.mode("overwrite").saveAsTable("ecommerce_bronze.customers")


### Read the order data and save in brone table

In [0]:
df_orders = spark.read.option("header", True).csv(
    "dbfs:/Volumes/dedatabricsws/demo/raw_volume/orders.csv"
)

df_orders.write.mode("overwrite").saveAsTable("ecommerce_bronze.orders")

In [0]:
df_orders.show()

### Read the product data and save it.

In [0]:
df_products = spark.read.option("header", True).csv(
    "dbfs:/Volumes/dedatabricsws/demo/raw_volume/products.csv"
)

df_products.write.mode("overwrite").saveAsTable("ecommerce_bronze.products")

### Read the payment data and save it.

In [0]:
df_payments = spark.read.option("header", True).csv(
    "dbfs:/Volumes/dedatabricsws/demo/raw_volume/payments.csv"
)

df_payments.write.mode("overwrite").saveAsTable("ecommerce_bronze.payments")

### 1 -ecommerce_bronze.customers
### 2 ecommerce_bronze.orders
### 3 ecommerce_bronze.products
### 4 ecommerce_bronze.payments

#### 🟠 STEP 3: SILVER LAYER (CLEAN + TRANSFORM)
#### 🎯 Goal
#### Clean nulls
#### Fix data types
#### Remove duplicates
#### Standardize schema

##### Customers Silver

In [0]:
from pyspark.sql.functions import col

In [0]:
df_orders = spark.table("dedatabricsws.ecommerce_bronze.customers")
df_orders.printSchema()
df_orders.show()

In [0]:


customers = spark.table("ecommerce_bronze.customers")

df_customers_silver = customers \
    .dropDuplicates() \
    .withColumn("customer_id", col("customer_id").cast("int"))

df_customers_silver.write.mode("overwrite").saveAsTable(
    "ecommerce_silver.customers"
)

In [0]:
df_orders = spark.table("dedatabricsws.ecommerce_silver.customers")
df_orders.printSchema()
df_orders.show()

##### Orders Silver

In [0]:
df_orders = spark.table("dedatabricsws.ecommerce_bronze.orders")
df_orders.printSchema()
df_orders.show()

In [0]:
orders = spark.table("ecommerce_bronze.orders")

df_orders_silver = orders \
    .dropDuplicates() \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("total_amount", col("total_amount").cast("double"))

df_orders_silver.write.mode("overwrite").saveAsTable(
    "ecommerce_silver.orders"
)

In [0]:
df_orders = spark.table("dedatabricsws.ecommerce_silver.orders")
df_orders.printSchema()
df_orders.show()

#### Products Silver

In [0]:
products = spark.table("ecommerce_bronze.products")

df_products_silver = products.dropDuplicates()

df_products_silver.write.mode("overwrite").saveAsTable(
    "ecommerce_silver.products"
)

#### payments silver


In [0]:
df_orders = spark.table("dedatabricsws.ecommerce_bronze.payments")
df_orders.printSchema()
#df_orders.show()

root
 |-- payment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: string (nullable = true)



In [0]:
payments = spark.table("ecommerce_bronze.payments")

df_payments_silver = payments \
    .dropDuplicates() \
    .withColumn("amount", col("amount").cast("double"))

df_payments_silver.write.mode("overwrite").saveAsTable(
    "ecommerce_silver.payments"
)

##### 🟢 RESULT SILVER LAYER
##### Clean datasets ready for analytics.

#### 🔵 STEP 4: GOLD LAYER (BUSINESS AGGREGATIONS)
##### 🎯 Goal Create analytics-ready tables

#### 4.1 Revenue by Customer

In [0]:
orders = spark.table("ecommerce_silver.orders")
payments = spark.table("ecommerce_silver.payments")

df_gold_customer_revenue = orders.join(
    payments,
    "order_id"
).groupBy("customer_id") \
 .sum("amount") \
 .withColumnRenamed("sum(amount)", "total_revenue")

df_gold_customer_revenue.write.mode("overwrite").saveAsTable(
    "ecommerce_gold.customer_revenue"
)

#### 4.2 Product Sales

In [0]:
df_gold_product_sales = orders.groupBy("product_id") \
    .count() \
    .withColumnRenamed("count", "total_orders")

df_gold_product_sales.write.mode("overwrite").saveAsTable(
    "ecommerce_gold.product_sales"
)

##### 4.3 Daily Revenue

In [0]:
from pyspark.sql.functions import to_date

df_daily_revenue = orders.join(payments, "order_id") \
    .withColumn("order_date", to_date("order_date")) \
    .groupBy("order_date") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "daily_revenue")

df_daily_revenue.write.mode("overwrite").saveAsTable(
    "ecommerce_gold.daily_revenue"
)

### 🟢 GOLD RESULT
#### Now you have BI-ready tables:

1 - customer_revenue
2 - product_sales
3 - daily_revenu

###### STEP 5: LOAD GOLD → SNOWFLAKE
#### 🎯 Goal Move curated data to Snowflake

#### Snowflake Connection (Databricks)

In [0]:


snowflake_options = {
  "host": "",
  "port": 443,
  "sfUser": "snowDream",
  "sfPassword":  "",
  "sfDatabase": "ECOMMERCE_DB",
  "sfSchema": "PUBLIC",
  "sfWarehouse": "COMPUTE_WH"
}



#### Write Gold Table to Snowflake

In [0]:
df_gold = spark.table("dedatabricsws.ecommerce_gold.customer_revenue")

df_gold.write \
  .format("snowflake") \
  .options(**snowflake_options) \
  .option("dbtable", "CUSTOMER_REVENUE") \
  .mode("overwrite") \
  .save()

In [0]:
df_gold = spark.table("dedatabricsws.ecommerce_gold.product_sales")

df_gold.write \
  .format("snowflake") \
  .options(**snowflake_options) \
  .option("dbtable", "PRODUCT_SALES") \
  .mode("overwrite") \
  .save()

In [0]:
df_gold = spark.table("dedatabricsws.ecommerce_gold.daily_revenue")

df_gold.write \
  .format("snowflake") \
  .options(**snowflake_options) \
  .option("dbtable", "DAILY_REVENUE") \
  .mode("overwrite") \
  .save()